# MobileBEV trên KITTI

Notebook này train từ dataset KITTI đã được `prepare_kitti.py` xử lý và lưu trên Google Drive.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

%cd /content
!test -d Lidar || git clone --branch mobileBEV-architecture --single-branch https://github.com/danhyoyo/Lidar.git
%cd /content/Lidar

!git branch --show-current
!git log -1 --oneline


In [ ]:
from pathlib import Path
import json
import subprocess

import torch

REPO_DIR = Path("/content/Lidar")
PROCESSED_DATASET_DIR = REPO_DIR / "data/kitti/processed"
ARTIFACT_ROOT = Path("/content/drive/MyDrive/mobilebev_artifacts")
RAW_KITTI_ROOT = Path("/content/KITTI_DATASET")

VARIANT = "A0"  # B0, A0, A1, A2, A3 hoặc A4
SEED = 42
PRECISION = "fp32"  # Dùng "bf16" nếu GPU hỗ trợ BF16
PHYSICAL_BATCH_SIZE = 2
ACCUMULATION_STEPS = 2
NUM_WORKERS = 2
EPOCHS = 100


## Copy dataset đã prepare từ Google Drive

Dataset cần có `pointcloud/`, `label/`, `train.txt` và `val.txt`. Raw KITTI không cần giải nén khi chỉ train.


In [ ]:
source_candidates = [
    Path("/content/drive/MyDrive/lidar/kitti_processed"),
    Path("/content/drive/MyDrive/lidar/kitti_processed/processed"),
]
PROCESSED_DATASET_SOURCE = next(
    (path for path in source_candidates if (path / "pointcloud").is_dir()),
    None,
)

if PROCESSED_DATASET_SOURCE is None:
    raise FileNotFoundError(
        "Không tìm thấy kitti_processed/pointcloud trên Google Drive."
    )

if not (PROCESSED_DATASET_DIR / "pointcloud").is_dir():
    PROCESSED_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["cp", "-a", f"{PROCESSED_DATASET_SOURCE}/.", str(PROCESSED_DATASET_DIR)],
        check=True,
    )

pointclouds = list((PROCESSED_DATASET_DIR / "pointcloud").glob("*.bin"))
labels = list((PROCESSED_DATASET_DIR / "label").glob("*.txt"))
assert len(pointclouds) == 7481, f"Pointcloud: {len(pointclouds)} != 7481"
assert len(labels) == 7481, f"Label: {len(labels)} != 7481"
assert not any(path.is_symlink() for path in pointclouds), (
    "Pointcloud đang là symlink; hãy tạo lại processed với --pointcloud-mode copy."
)

print(f"Dataset source: {PROCESSED_DATASET_SOURCE}")
print(f"Dataset local:  {PROCESSED_DATASET_DIR}")
print(f"Pointclouds: {len(pointclouds)}")
print(f"Labels: {len(labels)}")


## Cài thư viện và kiểm tra GPU


In [ ]:
%cd /content/Lidar
!pip install -q shapely onnx tqdm


In [ ]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU trong Runtime > Change runtime type.")

print("GPU:", torch.cuda.get_device_name(0))


## Kiểm tra code


In [ ]:
!MPLCONFIGDIR=/tmp/mobilebev-mpl python tests/test_mobile_bev.py


## Chọn variant và config

- `B0`: baseline loss.
- `A0`: UWAG + CoordAtt + augmentation.
- `A1`–`A4`: các biến thể MobileBEV.


In [ ]:
CONFIGS = {
    "A0": "configs/kitti/kitti_uwag_coordatt_aug.json",
    "A1": "configs/kitti/mobilebev/a1_legacy35_center3d.json",
    "A2": "configs/kitti/mobilebev/a2_rich8_center3d.json",
    "A3": "configs/kitti/mobilebev/a3_legacy35_sgfpn_center3d.json",
    "A4": "configs/kitti/mobilebev/a4_rich8_sgfpn_center3d.json",
    "A5": "configs/kitti/mobilebev/a5_rich11_center3d.json",
    "A6": "configs/kitti/mobilebev/a6_rich11_sgfpn_center3d.json",
}

if VARIANT == "B0":
    source_config = REPO_DIR / "configs/kitti/kitti_uwag_coordatt_aug.json"
    baseline_config = REPO_DIR / "configs/kitti/baseline_original.json"
    config = json.loads(source_config.read_text())
    config["loss"]["name"] = "baseline"
    baseline_config.write_text(json.dumps(config, indent=2) + "\n")
    CONFIG = str(baseline_config)
else:
    CONFIG = CONFIGS[VARIANT]

RUN_NAME = f"mobilebev_{VARIANT.lower()}_seed{SEED}"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

print("Variant:", VARIANT)
print("Config:", CONFIG)
print("Run:", RUN_NAME)


## Smoke test


In [ ]:
%cd /content/Lidar
!python tools/kitti_training_pipeline/train.py \
  --config "{CONFIG}" \
  --detector-root detector \
  --output-root "{ARTIFACT_ROOT}" \
  --run-name "{RUN_NAME}_smoke" \
  --device cuda \
  --precision "{PRECISION}" \
  --seed {SEED} \
  --epochs 1 \
  --physical-batch-size {PHYSICAL_BATCH_SIZE} \
  --accumulation-steps {ACCUMULATION_STEPS} \
  --max-train-batches 8 \
  --max-val-batches 4 \
  --num-workers {NUM_WORKERS}


## Train full


In [ ]:
%cd /content/Lidar
!python tools/kitti_training_pipeline/train.py \
  --config "{CONFIG}" \
  --detector-root detector \
  --output-root "{ARTIFACT_ROOT}" \
  --run-name "{RUN_NAME}" \
  --device cuda \
  --precision "{PRECISION}" \
  --seed {SEED} \
  --epochs {EPOCHS} \
  --physical-batch-size {PHYSICAL_BATCH_SIZE} \
  --accumulation-steps {ACCUMULATION_STEPS} \
  --num-workers {NUM_WORKERS}


## Chọn checkpoint

Với `B0`/`A0`, checkpoint được chọn theo validation loss. Với `A1`–`A4`, có thể chọn lại theo 3D mAP nếu cung cấp raw KITTI cho bước đánh giá.


In [ ]:
SELECT_3D_CHECKPOINT = False

if SELECT_3D_CHECKPOINT:
    if VARIANT in {"B0", "A0"}:
        raise ValueError("3D checkpoint selection chỉ áp dụng cho A1–A4.")
    if not (RAW_KITTI_ROOT / "training/label_2").is_dir():
        raise FileNotFoundError(
            "Bật SELECT_3D_CHECKPOINT cần raw KITTI có training/label_2 và training/calib."
        )

    subprocess.run([
        "python", "tools/kitti_training_pipeline/select_checkpoint.py",
        "--checkpoint-dir", str(ARTIFACT_ROOT / RUN_NAME / "checkpoints"),
        "--config", CONFIG,
        "--detector-root", "detector",
        "--kitti-root", str(RAW_KITTI_ROOT),
        "--split", "splits/kitti/val.txt",
        "--output-dir", str(ARTIFACT_ROOT / RUN_NAME / "selected_3d"),
        "--device", "cuda",
    ], check=True)
    CHECKPOINT = ARTIFACT_ROOT / RUN_NAME / "selected_3d/best.pt"
else:
    checkpoint_dir = "selected" if VARIANT in {"B0", "A0"} else "provisional_best_loss"
    CHECKPOINT = ARTIFACT_ROOT / RUN_NAME / checkpoint_dir / "best.pt"

print("Checkpoint:", CHECKPOINT)


## Đánh giá tùy chọn

Evaluator cần raw `label_2` và `calib` để tính ground truth. Có thể bỏ qua cell này nếu chỉ train.


In [ ]:
RUN_EVALUATION = False

if RUN_EVALUATION:
    if not (RAW_KITTI_ROOT / "training/label_2").is_dir():
        raise FileNotFoundError(
            "Đánh giá cần raw KITTI có training/label_2 và training/calib."
        )

    subprocess.run([
        "python", "tools/kitti_training_pipeline/evaluate_kitti_bev.py",
        "--name", RUN_NAME,
        "--backend", "pytorch",
        "--model", str(CHECKPOINT),
        "--config", CONFIG,
        "--detector-root", "detector",
        "--kitti-root", str(RAW_KITTI_ROOT),
        "--split", "splits/kitti/val.txt",
        "--output", str(ARTIFACT_ROOT / RUN_NAME / "evaluation.json"),
        "--device", "cuda",
        "--warmup-frames", "10",
    ], check=True)
else:
    print("Đã bỏ qua evaluation.")
